# Does mediation reduce battle intensity in civil wars?

Many civil wars experience early third-party efforts to assist negotiations. Mediation can contribute directly to a reduction
of violence in two ways. First, mediators can ease the path to negotiated settlements by facilitating the flow of information through communication and fact-finding. Second, the negotiation process itself creates short-term space for temporary ceasefires. Therefore, I would like to test whether the presence of third-party mediation in civil wars is correlated with a reduced level of battle intensity, which will be reflected by the battle-related deaths. 

I was inspired by an academic paper on mediation/peacekeeping and the civil war severity and wanted to conduct a simplified replication of the analysis without using any statistical models: Beardsley, K., Cunningham, D. E., & White, P. B. (2018). Mediation, Peacekeeping, and the Severity of Civil War. Journal of Conflict Resolution, 63(7), 1682-1709. https://doi.org/10.1177/0022002718817092 (Original work published 2019)

- **Dataset(s) to be used:** 
  - [Dataset 1 on mediation] [https://ucdp.uu.se/downloads/index.html#mic]
  - [Dataset 2 on battle intensity] [https://ucdp.uu.se/downloads/index.html#battlerelated]
- **Analysis question:** [Does the presence of mediation in civil wars reduce battle-related deaths?]
- **Columns to be used to merge/join them:**
  - [Dataset 1] [thid_states_no, third_igo_no, third_other, start_event, conflict_name]
  - [Dataset 2] [bd_best, year, battle_location]
- **Hypothesis**: [The presence of third-party mediation in civil wars will result in reduced number of battle-related deaths.]
  - Hypothesis 1: The presence of third-party mediation in civil wars will result in reduced number of battle-related deaths.
  - Hypothesis 2: The more third-party mediators involved, the fewer battle-related deaths will be.
  - Hypothesis 3: The more state mediators involved, the fewer battle-related deaths will be.
  - Hypothesis 4: The more IGO mediators involved, the fewer battle-related deaths will be.
  - Hypothesis 5: The more other mediators involved, the fewer battle-related deaths will be.

In [1]:
# ensure the visualizations render properly across VSCode, Jupyter Book, etc.
# https://plotly.com/python/renderers/

import plotly.io as pio

pio.renderers.default = "notebook_connected+plotly_mimetype"

# About the Data

Both datasets contain dyadic data on conflict parties. To simplify the analysis, I only refer to the place where the third-party mediation and the battle/conflict occur. <br>
- **Mediation dataset:** 
    - Data source: Uppsala Conflict Data Program (UCDP)
    - Temporal indicator: start_event refers to the date when event begins, given in the standard ISO format (YYYY-MM-DD). 
    - Mediation:  
        - third_states_no: Indicates number of individual states involved in the third party event. 
        - third_igo_no: Indicates number of international governmental organizations involved in the third party event.
        - third_other: Indicates number of third parties involved in the third party event, which are neither a state nor an IGO. e.g. private person, NGO, religious denomination.
        - I will also create a new column "third_no", aggregating all the numbers of different third parties.
- **Battle-related deaths dataset:**
    - Data sources: Uppsala Conflict Data Program (UCDP)
    - Time: the column "year" will be used. However, since the unit of temporal data in this dataset is year, I will resample the temporal data in the first dataset on mediation to the unit of year. To avoid confusion in the sequence of mediation and battle intensity, I will lag the year of battle-related death by one when conducting the visualization and analysis.
    - bd_best: Refers to the UCDP Best estimate for battle-related deaths in the conflict dyad in the given year.

In [2]:
import pandas as pd
import plotly.express as px

In [3]:
# Read dataset 1: Mediation
Mediation = pd.read_csv("mic.csv")


In [4]:
# Read dataset 2: Battle-related deaths
Battle = pd.read_csv("BattleDeaths.csv")

# Data Cleaning and Merging


- I will clean the columns by only keeping the columns that will be used to merge and analyze.
- As mentioned in the previous section, I will create a new column in dataset 1(Mediation) to aggregate the total number of all third-parties.
- I will also turn the Mediation data into a DateTime and resample the date to organize mediation events by year, in alignment with the unit of data (year) in dataset 2 (Battle).

### Clean Dataset 1: Mediation

In [5]:
# Clean the columns to keep columns for analysis
Mediation = Mediation[
    ["conflict_name", "start_event", "third_states_no", "third_igo_no", "third_other"]
]

Mediation = Mediation.rename(
    columns={
        "conflict_name": "Conflict Name",
        "start_event": "Mediation Date",
        "third_states_no": "State Mediators",
        "third_igo_no": "IGO Mediators",
        "third_other": "Other Mediators",
    }
)


In [ ]:
# Create a new column to for the total number of third party mediators
cols = ["State Mediators", "IGO Mediators", "Other Mediators"]
Mediation["Total Mediators"] = Mediation[cols].fillna(0).sum(axis=1)


In [7]:
# Convert to DateFrame and Resample
Mediation["Mediation Date"] = pd.to_datetime(
    Mediation["Mediation Date"], format="%m/%d/%Y"
)

# Extract year
Mediation["Year"] = Mediation["Mediation Date"].dt.year

# Resample by year (need datetime index)
Mediation = Mediation.set_index("Mediation Date")

# Count events per year
yearly_mediation = Mediation.resample("Y").size()

# Convert index to YYYY
yearly_mediation.index = yearly_mediation.index.year

Mediation["Conflict Name"] = Mediation["Conflict Name"].astype(str)


/var/folders/hy/_6n5b67s1b3565tszxpc131c0000gn/T/ipykernel_43589/2431648424.py:13: FutureWarning:

'Y' is deprecated and will be removed in a future version, please use 'YE' instead.



### Clean Dataset 2: Battle-Related Deaths

In [8]:
# Clean the columns
Battle = Battle[["year", "bd_best", "battle_location"]]

Battle = Battle.rename(
    columns={
        "year": "Year",
        "bd_best": "Deaths Estimate",
        "battle_location": "Conflict Name",
    }
).sort_values(["Year"])


### Merge the two datasets

- I will firstly merge the two dataset based on Conflict Name and Year, and then fill all NaN values with 0.
- Due to time inconsistensy in the two datasets, I will lag all Years of the Battle Deaths by one year and create a new column named "Deaths in the Next Year". In this way, we can draw the correlation between mediation and battle intensity more accurately.

In [9]:
# Merge the two datasets based on year and conflict name
Merged_Data = pd.merge(
    Mediation,
    Battle,
    on=["Conflict Name", "Year"],
    how="outer",
)

Merged_Data.head()

,Conflict Name,State Mediators,IGO Mediators,Other Mediators,Total Mediators,Year,Deaths Estimate
0,Afghanistan,NaN,NaN,NaN,NaN,1989,5174.0
1,Afghanistan,NaN,NaN,NaN,NaN,1990,1478.0
2,Afghanistan,NaN,NaN,NaN,NaN,1991,3302.0
3,Afghanistan,NaN,NaN,NaN,NaN,1992,4276.0
4,Afghanistan,NaN,NaN,NaN,NaN,1993,3721.0


In [10]:
# Convert the types of Conflict Name and Year respectively to string and integer
Merged_Data["Conflict Name"] = Merged_Data["Conflict Name"].astype(str)

#  Sort first
Merged_Data = Merged_Data.sort_values(["Conflict Name"])

# Create next-year deaths within each conflict
Merged_Data["Deaths in the Next Year"] = Merged_Data.groupby("Conflict Name")[
    "Deaths Estimate"
].shift(-1)
Merged_Data = Merged_Data.dropna(subset=["Deaths in the Next Year"])


Merged_Data.info()
Merged_Data.head()


<class 'pandas.core.frame.DataFrame'>
Index: 4088 entries, 0 to 5047
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Conflict Name            4088 non-null   object 
 1   State Mediators          2799 non-null   float64
 2   IGO Mediators            2799 non-null   float64
 3   Other Mediators          2799 non-null   float64
 4   Total Mediators          2799 non-null   float64
 5   Year                     4088 non-null   int64  
 6   Deaths Estimate          4003 non-null   float64
 7   Deaths in the Next Year  4088 non-null   float64
dtypes: float64(6), int64(1), object(1)
memory usage: 287.4+ KB


,Conflict Name,State Mediators,IGO Mediators,Other Mediators,Total Mediators,Year,Deaths Estimate,Deaths in the Next Year
0,Afghanistan,NaN,NaN,NaN,NaN,1989,5174.0,105.0
30,Afghanistan,NaN,NaN,NaN,NaN,2024,105.0,28.0
29,Afghanistan,NaN,NaN,NaN,NaN,2024,28.0,74.0
28,Afghanistan,NaN,NaN,NaN,NaN,2023,74.0,149.0
27,Afghanistan,NaN,NaN,NaN,NaN,2023,149.0,253.0


# Visualization

- In this part, I will draw visualizations of the independent variables we have regarding mediation, and the dependent variable of battle-related deaths in the next year.
- I choose to use scatter plots together with an additional OLS line for non-binary variables, to reflect the correlation.
- One limitations of scatter plots is that it cannot include all conflict/country names as part of the visualization, otherwise the plots will look quite messy even if I use different colors to indicate different countries, due to the sheer volume of them. Therefore, I choose to add a separate plot - selecting five different countries (Conflict Name) and draw the visualization for mediation and battle intensity in these five countries at the end.


### Figure 1: The Influence on the Next-Year Deaths by Presence of Mediators

In [11]:
import plotly.express as px

Merged_Data["Has_Mediators"] = (Merged_Data["Total Mediators"] > 0).astype(str)

fig = px.scatter(
    Merged_Data,
    x="Has_Mediators",
    y="Deaths in the Next Year",
    opacity=0.6,
    labels={
        "Has_Mediators": "Total Mediators > 0",
        "Deaths in the Next Year": "Next-Year Battle Deaths",
    },
    title="Figure 1: The Influence on the Next-Year Deaths by Presence of Mediators",
)

fig.show()

### Figure 2: The Influence of All Mediation on Next-Year Battle Deaths

In [12]:
import plotly.express as px

fig = px.scatter(
    Merged_Data,
    x="Total Mediators",
    y="Deaths in the Next Year",
    opacity=0.5,
    trendline="ols",
    labels={
        "Total Mediators": "Total Mediators",
        "Deaths in the Next Year": "Next-Year Battle Deaths",
    },
    title="Figure 2: The Influence of All Mediation on Next-Year Battle Deaths",
    range_y=[0, 6000],
    range_x=[0, 15],
)
fig.show()


### Figure 3: The Influence of State Mediation on Next-Year Battle Deaths

In [13]:
import plotly.express as px

fig = px.scatter(
    Merged_Data,
    x="State Mediators",
    y="Deaths in the Next Year",
    opacity=0.5,
    trendline="ols",
    labels={
        "State Mediators": "State Mediators",
        "Deaths in the Next Year": "Next-Year Battle Deaths",
    },
    title="Figure 3: The Influence of State Mediation on Next-Year Battle Deaths",
    range_y=[0, 6000],
    range_x=[0, 10],
)
fig.show()

### Figure 4: The Influence of IGO Mediation on Next-Year Battle Deaths

In [14]:
import plotly.express as px

fig = px.scatter(
    Merged_Data,
    x="IGO Mediators",
    y="Deaths in the Next Year",
    opacity=0.5,
    trendline="ols",
    labels={
        "IGO Mediators": "IGO Mediators",
        "Deaths in the Next Year": "Next-Year Battle Deaths",
    },
    title="Figure 4: The Influence of IGO Mediation on Next-Year Battle Deaths",
    range_y=[0, 6000],
    range_x=[0, 4],
)
fig.show()

### Figure 5: The Influence of Other Mediation on Next-Year Battle Deaths

In [15]:
import plotly.express as px

fig = px.scatter(
    Merged_Data,
    x="Other Mediators",
    y="Deaths in the Next Year",
    opacity=0.8,
    trendline="ols",
    labels={
        "Other Mediators": "Other Mediators",
        "Deaths in the Next Year": "Next-Year Battle Deaths",
    },
    title="Figure 5: The Influence of Other Mediation on Next-Year Battle Deaths",
    range_y=[0, 10000],
)
fig.show()

### Figure 6: The Influence of the Number of All Mediators on Next-Year Battle Deaths in Five Countries

In [16]:
import plotly.express as px

selected_countries = ["Algeria", "Liberia", "Angola", "Sierra Leone", "Sudan"]

Merged_Data_filtered = Merged_Data[
    Merged_Data["Conflict Name"].isin(selected_countries)
]

fig = px.scatter(
    Merged_Data_filtered,
    x="Total Mediators",
    y="Deaths in the Next Year",
    opacity=0.5,
    color="Conflict Name",
    trendline="ols",
    trendline_scope="trace",
    labels={
        "Total Mediators": "Total Mediators",
        "Deaths in the Next Year": "Next-Year Battle Deaths",
    },
    title="Figure 6: The Influence of the Number of All Mediators on Next-Year Battle Deaths in Five Countries",
    range_y=[0, 4000],
    range_x=[0, 20],
)
fig.show()

# Results

 First, let's review our hypotheses in the beginning regarding the influence of mediation on battle intensity: 
- Hypothesis 1: The presence of third-party mediation in civil wars will result in reduced number of battle-related deaths.
- Hypothesis 2: The more third-party mediators involved, the fewer battle-related deaths will be.
- Hypothesis 3: The more state mediators involved, the fewer battle-related deaths will be.
- Hypothesis 4: The more IGO mediators involved, the fewer battle-related deaths will be.
- Hypothesis 5: The more other mediators involved, the fewer battle-related deaths will be.

From all five figures, we can tentatively confirm hypothesis 1, 2 and 4. The presence of third-party mediation in civil wars could create an environment conducive for less intense battles. Meanwhile, the total number of mediators, particularly IGO mediators, tend to produce better effects at reducing battle-related deaths and lowering the battle severity.

The reason why state mediators might not generate better outcomes could be that regional actors sometimes are involved in complex relationships with the conflict parties and might even serve as patrons to some of them, thereby spoiling rather than contributing to the mediation outcomes.

The scatter plot involving five selected countries demonstrate that mediation has varied outcomes in different conflict settings. It is thus significant to find out other compounding factors that influence the effects of mediation, such as the deployment of peacekeeping forces, external support to conlict parties, and so on.

# Thank you for reading!